**Human Movement Intelligence | Biomechanical Load Estimation | Presented by: Reza Saadatyar (2026)**

**06 | Deep Temporal Regression and Biomechanical Peak Prediction**

![Essex temporal regression study, seven source wearable IMUs, native FE alignment, participant-disjoint model development and reserved Test](../../images/06_consolidated_workflow.svg)

**Research question:** Can compact temporal neural networks improve prediction of original first (`e1`) and third (`e3`) principal femoral-neck finite-element (FE) strain curves from the complete six-axis, seven-IMU trial signals, compared with the Stage 05 classical baselines, without compromising participant-level separation?

**Data foundation:** The checksum-verified [Essex Figshare v13 source](https://doi.org/10.6084/m9.figshare.30225739.v13), audited in [Notebook 04](04_governed_imu_fe_dataset_and_participant_splits.ipynb) and benchmarked in [Notebook 05](05_classical_regression_baselines_and_participant_independent_validation.ipynb). All 1,729 source-eligible trials retain native `101×42` IMU and `11×2` FE targets. Only Training (29 people, 1,354 trials) and Validation (four people, 192 trials) enter this study. The four Test subjects (183 trials) remain reserved.

**Temporal alignment:** Convolutional features are computed at all 101 recorded IMU coordinates; a source-verified index map selects the **eleven exact original FE positions**. The models predict two strain values at each of these positions. No interpolation, upsampling or invented FE measurements are used.

**Fixed model comparison:** Temporal CNN versus residual dilated TCN, each trained with the same 35-epoch budget, fold-specific input/target standardisation, AdamW optimisation and predeclared waveform-plus-peak objective. Five previously saved GroupKFold assignments from Training are reused. Model selection uses Training out-of-fold subject-balanced MAE **only**; independent predetermined Validation provides diagnostic results, not selection.

**Scientific interpretation:** The published FE strain outputs are **unitless model-derived targets**, not direct clinical measurements. Peak magnitude and peak-index differences refer to the eleven original FE coordinates. Stage 05 Random Forest and Ridge results provide recorded context, but their prior inspection means the programme's Validation set is **not fully blind for cross-stage research decisions**. Do not claim independent external validation or prospective deployment.

**Execution:** The simplified cells have not been rerun on the full dataset. Numerical findings described here refer to the previously completed run. Each notebook reads the current upstream files; re-execution replaces its own output files. No historical reports, hash comparisons or PASSED gates are required.


**00. Colab Repository Setup**


In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

Repository up to date.
/content/human-movement-intelligence


**01. Imports and Research Environment**


In [ ]:
# ==================== 01. Imports and Research Environment ===================

# ------------------------- 1. Study Environment Setup ------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("biomechanical_load_estimation")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd
import torch

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
plot_style(font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5, label_weight="normal",
    tick_size=11, legend_size=10, figure_dpi=100, figure_size=(9, 4))

# ------------------- 5. Shared and Study Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from essex_regression_baselines import (
    regression_metrics, participant_metrics, subject_balanced_mae,
    trial_predictions,
)
from essex_temporal_regression import MODEL_NAMES, fit_temporal_model
from essex_paired_dataset import (
    read_trial_splits,
    load_paired_data,
    read_training_folds,
)

# ----------------------- 6. Source and Output Locations ----------------------
study_dir = root / "biomechanical_load_estimation"
raw_dir = root / "data" / "raw" / "essex_femoral_neck_loading"
mat_dir = raw_dir / "matlab_records"
processed_dir = root / "data" / "processed" / "essex_femoral_neck_loading"
output_dir = study_dir / "output"
processed_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Deep learning device:", device)
print("Stage 06 evidence directory:", output_dir)


**02. Current Participant Splits and Model Inputs**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# ======================= 02. Current Participant Splits ======================

# --------------------- 1. Read Current Trial Assignments ---------------------
assigned = read_trial_splits(output_dir)

# --------------------- 2. Prepare Participant Split Masks --------------------
train_mask = assigned["split"].eq("TRAIN").to_numpy()
valid_mask = assigned["split"].eq("VALIDATION").to_numpy()
test_mask = assigned["split"].eq("TEST").to_numpy()
display(assigned.groupby("split").agg(
    trials=("local_record_key", "size"),
    participants=("subject_id", "nunique"),
))

# ------------------- 3. Read Current Upstream Model Results ------------------
base_cv_path = output_dir / "05_training_cv_model_summary.csv"
base_val_path = output_dir / "05_validation_model_metrics.csv"
fold_path = output_dir / "05_training_fold_registry.csv"
baseline_cv = pd.read_csv(base_cv_path)
baseline_val = pd.read_csv(base_val_path)
prior_cv_score = float(baseline_cv.loc[
    baseline_cv["model"].eq("Random Forest"), "subject_balanced_mae"
].iloc[0])
prior_validation_score = float(baseline_val.loc[
    baseline_val["model"].eq("Random Forest"),
    "subject_balanced_mae",
].iloc[0])


**03. Load Current Paired Arrays**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# ======================= 03. Load Current Paired Arrays ======================

# --------------- 1. Install and Import the Pinned MATLAB Reader --------------
%pip -q install --only-binary=:all: --no-deps "mat-io==1.0.1"
from matio import load_from_mat

# ----------------------- 2. Load Current Paired Arrays -----------------------
paired = load_paired_data(output_dir, processed_dir, raw_dir, load_from_mat, assigned)

# ------------ 3. Select Arrays for the Required Participant Splits -----------
x_train_source = paired["X_full"][train_mask]
y_train_source = paired["y_fe"][train_mask]
x_valid_source = paired["X_full"][valid_mask]
y_valid_source = paired["y_fe"][valid_mask]
idx_train_source = paired["imu_positions"][train_mask]
idx_valid_source = paired["imu_positions"][valid_mask]
print("Loaded current arrays:", {name: value.shape for name, value in paired.items()})


**04. Frozen Training Fold Registry and Temporal Data Contract**

The 1,354 Training trials and **five participant-disjoint fold assignments** were reused exactly from Notebook 05. Training contained 29 participants; the independent diagnostic Validation partition contained four participants and 192 trials. Four further Test participants (183 trials) remained reserved. The input and target contracts were the original 101-by-42 IMU signal and 11-by-2 source-aligned FE output, without target interpolation.


In [ ]:
# ================== 04. Native-Time Trial and Split Contract =================

# ------------------------- 1. Fixed Trial Identities -------------------------
train_info = assigned.loc[train_mask].reset_index(drop=True)
valid_info = assigned.loc[valid_mask].reset_index(drop=True)
reserved_ids = set(assigned.loc[test_mask, "subject_id"])
train_ids = set(train_info["subject_id"])
valid_ids = set(valid_info["subject_id"])

# ------------------ 2. Reuse Stage 05 Participant GroupKFold -----------------
prior_folds = read_training_folds(output_dir, train_info)
TRAIN_FOLDS = 5
SEED = 2026
EPOCHS = 35
BATCH = 64
PEAK_WEIGHT = 0.25
LEARNING_RATE = 1e-3
fold_labels = prior_folds["validation_fold"].to_numpy()
groups = train_info["subject_id"].to_numpy()
folds = []
for fold_id in range(1, TRAIN_FOLDS + 1):
    fit_rows = np.flatnonzero(fold_labels != fold_id)
    held_rows = np.flatnonzero(fold_labels == fold_id)
    folds.append((fit_rows, held_rows))

# ------------ 3. Prepare Dataset Arrays and Save Fold Assignments ------------
y_train = y_train_source
y_valid = y_valid_source
x_train = x_train_source
x_valid = x_valid_source
idx_train = idx_train_source
idx_valid = idx_valid_source
prior_folds.to_csv(
    output_dir / "06_frozen_training_fold_registry.csv", index=False,
)
print("Frozen five-fold registry reused exactly from Notebook 05.")
print("Source-aligned temporal inputs:", x_train.shape)
print("Reserved Test subjects (not fitted or scored):", len(reserved_ids))
display(prior_folds.groupby("validation_fold").agg(
    trials=("local_record_key", "size"),
    participants=("subject_id", "nunique"),
))


**05. Matched Training-Only Deep Learning Cross-Validation**

Both Temporal CNN and Dilated TCN completed five Training-only grouped fits, each using a fixed 35-epoch budget, fit-fold-only standardisation, AdamW and the same predeclared normalised waveform-plus-peak loss (peak weight 0.25). Ten fold fits and two final Training fits were recorded. These are single-seed-schedule experiments, not repeated-seed robustness or exhaustive hyperparameter tuning. The saved environment used CPU.


In [ ]:
# ===================== 05. Matched Grouped Model Training ====================

# ----------------------- 1. Predeclared Training Budget ----------------------
print("Temporal models:", MODEL_NAMES)
print("Training-fold fits:", len(MODEL_NAMES) * TRAIN_FOLDS)
print("Epochs per fit:", EPOCHS)
print("No early stopping and no Test-set access.")

# ----------------- 2. Train on Each Participant-Disjoint Fold ----------------
oof_prediction = {}
fold_metric_rows = []
history_frames = []
parameter_counts = {}
for model_name in MODEL_NAMES:
    oof = np.full(y_train.shape, np.nan, dtype=np.float64)
    for fold_number, (fit_rows, held_rows) in enumerate(folds, 1):
        prediction, history, facts = fit_temporal_model(
            model_name,
            x_train[fit_rows],
            y_train[fit_rows],
            idx_train[fit_rows],
            x_train[held_rows],
            idx_train[held_rows],
            seed=SEED + fold_number,
            epochs=EPOCHS,
            batch_size=BATCH,
            learning_rate=LEARNING_RATE,
            peak_weight=PEAK_WEIGHT,
            device=device,
        )
        oof[held_rows] = prediction
        parameter_counts[model_name] = facts["trainable_parameters"]
        history["model"] = model_name
        history["fold"] = fold_number
        history["fit_trials"] = len(fit_rows)
        history_frames.append(history)
        balanced = subject_balanced_mae(
            y_train[held_rows], prediction, groups[held_rows]
        )
        for result in regression_metrics(y_train[held_rows], prediction):
            fold_metric_rows.append(
                {
                    "model": model_name,
                    "fold": fold_number,
                    "fit_participants": len(set(groups[fit_rows])),
                    "held_participants": len(set(groups[held_rows])),
                    "subject_balanced_mae": balanced,
                    **result,
                }
            )
        print(
            model_name,
            "| fold",
            fold_number,
            "| subject-balanced MAE:",
            f"{balanced:.7f}",
        )
    oof_prediction[model_name] = oof
history_all = pd.concat(history_frames, ignore_index=True)
cv_fold_metrics = pd.DataFrame(fold_metric_rows)
history_all.to_csv(
    output_dir / "06_training_loss_history.csv", index=False,
)
cv_fold_metrics.to_csv(
    output_dir / "06_training_cv_fold_metrics.csv", index=False,
)
display(cv_fold_metrics.head(10))


**06. Training Out-of-Fold Selection and Peak Diagnostics**

**Dilated TCN** achieved the lower predeclared Training out-of-fold subject-balanced MAE: **0.0008513**, compared with **0.0009048** for Temporal CNN, and was therefore selected before the Stage 06 Validation analysis. Its Training out-of-fold e1 and e3 pointwise MAEs were **0.0005192** and **0.0011806**. The Stage 05 Random Forest reference remained better on the overall Training-CV criterion (**0.0007950**).


In [ ]:
# ================== 06. Training CV Results and Model Choice =================

# ------------------ 1. Participant and Activity-Level Errors -----------------
summary_rows = []
cv_diagnostics = []
for model_name in MODEL_NAMES:
    prediction = oof_prediction[model_name]
    diagnostic = participant_metrics(
        y_train, prediction, train_info["subject_id"],
        train_info["task"], model=model_name, phase="TRAIN_OOF",
    )
    diagnostic["fold"] = diagnostic["subject_id"].map(
        prior_folds.drop_duplicates("subject_id").set_index(
            "subject_id"
        )["validation_fold"]
    ).astype(int)
    cv_diagnostics.append(diagnostic)
    by_target = regression_metrics(y_train, prediction)
    summary_rows.append({
        "model": model_name,
        "subject_balanced_mae": subject_balanced_mae(
            y_train, prediction, train_info["subject_id"],
        ),
        "e1_oof_mae": by_target[0]["mae"],
        "e3_oof_mae": by_target[1]["mae"],
        "e1_peak_mae": by_target[0]["peak_value_mae"],
        "e3_peak_mae": by_target[1]["peak_value_mae"],
        "training_subjects": len(train_ids),
        "training_trials": len(train_info),
        "cv_folds": TRAIN_FOLDS,
    })
cv_details = pd.concat(cv_diagnostics, ignore_index=True)
cv_summary = pd.DataFrame(summary_rows).sort_values(
    ["subject_balanced_mae", "model"], kind="stable",
).reset_index(drop=True)
selected_temporal = str(cv_summary.iloc[0]["model"])
cv_details.to_csv(
    output_dir / "06_training_cv_subject_activity_metrics.csv", index=False,
)
cv_summary.to_csv(
    output_dir / "06_training_cv_model_summary.csv", index=False,
)
print("Selected using Training-only OOF errors:", selected_temporal)
display(cv_summary)


**07. Fixed Full-Training Fits and Diagnostic Validation**

Both neural architectures were refitted using all 29 Training participants and the same fixed training budget. On the four predefined Validation participants (192 trials), the Training-selected **Dilated TCN** achieved subject-balanced MAE **0.0007151**, versus **0.0007517** for Temporal CNN. Dilated TCN pointwise MAE was **0.0003975** for e1 and **0.0010328** for e3. These Validation values were descriptive, not a retrospective model-selection criterion.


In [ ]:
# =================== 07. Source-Held Validation Diagnostics ==================

# ---------------------- 1. Fit Training-Only Full Models ---------------------
validation_predictions = {}
val_rows = []
prediction_frames = []
full_histories = []
for model_number, model_name in enumerate(MODEL_NAMES, 1):
    prediction, history, facts = fit_temporal_model(
        model_name, x_train, y_train, idx_train,
        x_valid, idx_valid,
        seed=SEED + 100 + model_number,
        epochs=EPOCHS, batch_size=BATCH,
        learning_rate=LEARNING_RATE, peak_weight=PEAK_WEIGHT,
        device=device,
    )
    validation_predictions[model_name] = prediction
    history["model"] = model_name
    history["fold"] = 0
    history["fit_trials"] = len(x_train)
    full_histories.append(history)
    balanced = subject_balanced_mae(
        y_valid, prediction, valid_info["subject_id"],
    )
    for row in regression_metrics(y_valid, prediction):
        val_rows.append({
            "model": model_name,
            "selected_from_training_cv": model_name == selected_temporal,
            "subject_balanced_mae": balanced,
            "validation_participants": len(valid_ids),
            **row,
        })
    prediction_frames.append(trial_predictions(
        y_valid, prediction,
        valid_info["local_record_key"], valid_info["subject_id"],
        valid_info["task"], model_name,
    ))
val_metrics = pd.DataFrame(val_rows)
val_predictions = pd.concat(prediction_frames, ignore_index=True)
hist_full = pd.concat(full_histories, ignore_index=True)
history_all = pd.concat([history_all, hist_full], ignore_index=True)
history_all.to_csv(
    output_dir / "06_training_loss_history.csv", index=False,
)
val_metrics.to_csv(
    output_dir / "06_validation_model_metrics.csv", index=False,
)
val_predictions.to_csv(
    output_dir / "06_validation_predictions.csv", index=False,
)
print("Training-selected temporal model:", selected_temporal)
print("Validation results are diagnostic, not a selection criterion.")
display(val_metrics)


**08. Biomechanical Error and Stage 05 Comparison**

**Waveform prediction:** The Stage 05 Random Forest remained better in overall Validation subject-balanced MAE (**0.0006771**) than Dilated TCN (**0.0007151**), representing approximately **5.6% higher** error for the neural model.

**Peak-magnitude prediction:** Dilated TCN reduced Validation peak-value MAE relative to Random Forest for e1 (**0.0005855** versus **0.0006625**, approximately **11.6% lower**) and e3 (**0.0014333** versus **0.0016423**, approximately **12.7% lower**).

**Peak location:** The advantage did not extend to peak-position error. Dilated TCN averaged **1.615** and **1.495** original FE indices for e1 and e3, compared with **1.417** and **1.005** for Random Forest. Peak magnitude and peak timing are separate scientific outcomes. Cross-stage Validation comparisons remain descriptive because these same four participants were previously inspected during research development.


In [ ]:
# =============== 08. Biomechanical and Cross-Stage Diagnostics ===============

# ----------------- 1. Original Participant/Activity Evidence -----------------
val_details = pd.concat([
    participant_metrics(
        y_valid, validation_predictions[model_name],
        valid_info["subject_id"], valid_info["task"],
        model=model_name, phase="VALIDATION",
    ) for model_name in MODEL_NAMES
], ignore_index=True)
val_details.to_csv(
    output_dir / "06_validation_subject_activity_metrics.csv", index=False,
)

# ------------------ 2. Preserve Earlier Classical Benchmark ------------------
context05 = baseline_val.loc[:, [
    "model", "target", "subject_balanced_mae", "mae", "rmse",
    "peak_value_mae", "peak_index_mae", "trials",
]].copy()
context05["study_stage"] = "05_CLASSICAL_RECORDED"
context06 = val_metrics.loc[:, [
    "model", "target", "subject_balanced_mae", "mae", "rmse",
    "peak_value_mae", "peak_index_mae", "trials",
]].copy()
context06["study_stage"] = "06_TEMPORAL_RECORDED"
comparison = pd.concat([context05, context06], ignore_index=True)
comparison.to_csv(
    output_dir / "06_validation_cross_stage_comparison.csv",
    index=False,
)
print("Baseline Random Forest CV MAE (recorded):", f"{prior_cv_score:.7f}")
print("Baseline Random Forest Validation MAE (recorded):",
      f"{prior_validation_score:.7f}")
print("Compare descriptive errors; do not select a model from Validation.")
display(comparison)


**09. Scientific Figures and Source-Identified Waveform Examples**

The executed outputs contain two grouped-bar comparisons (Validation waveform MAE and peak-magnitude MAE) and original 11-position e1/e3 curves for the fixed Validation example **subj10:0000**. This is an illustrative source-identified trial, not representative evidence of performance across all participants. Original FE points are neither interpolated nor counted as independent trials.


In [ ]:
# =================== 09. Scientific Model and Peak Figures ===================

# ----------------------- 1. Paired Target-Level Errors -----------------------
order = ["Training Mean", "Ridge", "Random Forest", *MODEL_NAMES]
curve = comparison.pivot(
    index="model", columns="target", values="mae",
).reindex(order)

# ------------------- 2. Peak-Magnitude Errors Are Distinct -------------------
fig, ax = plt.subplots(figsize=(10, 4))
curve.plot.bar(ax=ax, width=0.75)
ax.set_title("Validation waveform MAE: classical and temporal models")
ax.set_xlabel("Model (cross-stage diagnostic comparison)")
ax.set_ylabel("MAE (published unitless FE scale)")
ax.tick_params(axis="x", rotation=15)
save_figure(fig, output_dir / "06_validation_waveform_mae_comparison.png")
plt.show()
peak = comparison.pivot(
    index="model", columns="target", values="peak_value_mae",
).reindex(order)
fig, ax = plt.subplots(figsize=(10, 4))
peak.plot.bar(ax=ax, width=0.75)
ax.set_title("Validation FE peak-magnitude MAE by model and target")
ax.set_xlabel("Model (not a retrospective selection criterion)")
ax.set_ylabel("Peak value MAE (unitless FE scale)")
ax.tick_params(axis="x", rotation=15)
save_figure(fig, output_dir / "06_validation_peak_mae_comparison.png")
plt.show()

# ---------------- 3. Genuine Source-Key-Identified FE Examples ---------------
example_i = int(np.argsort(
    valid_info["local_record_key"].to_numpy()
)[0])
example_key = str(valid_info.iloc[example_i]["local_record_key"])
print("Fixed illustrative Validation trial:", example_key)
for j, target in enumerate(("e1", "e3")):
    fig, ax = plt.subplots(figsize=(9, 4))
    positions = np.arange(11)
    ax.plot(
        positions, y_valid[example_i, :, j],
        marker="o", label="Original source FE",
    )
    for model_name in MODEL_NAMES:
        ax.plot(
            positions,
            validation_predictions[model_name][example_i, :, j],
            marker="s" if model_name == MODEL_NAMES[0] else "^",
            label=model_name,
        )
    ax.set_title(f"Validation example: {target} | {example_key}")
    ax.set_xlabel("Original FE grid index (0–10)")
    ax.set_ylabel("Published unitless FE strain")
    ax.set_xticks(positions)
    ax.legend()
    save_figure(fig, output_dir / f"06_validation_waveform_{target}.png")
    plt.show()


**10. Results Summary and Output Download**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# ================== 10. Results Summary and Output Download ==================

# ------------------------- 1. Display Current Results ------------------------
print("Current temporal-model results saved in output/ with prefix 06_.")
print("Training trials:", len(train_info), "| validation trials:", len(valid_info))

# -------------------- 2. Download Current Notebook Outputs -------------------
download_outputs_from_colab(
    folder="biomechanical_load_estimation/output", stage="06",
)


**11. Scientific Interpretation and Next Research Boundary**

**Completed model comparison.** Dilated TCN was selected over Temporal CNN using Training-only five-fold, participant-disjoint out-of-fold MAE (**0.0008513** versus **0.0009048**). Its Validation MAE was **0.0007151**, compared with **0.0007517** for Temporal CNN, on four participants and 192 trials. All network fits used the predeclared 35-epoch budget and the recorded execution device was CPU.

**Classical versus deep learning.** The Stage 05 Random Forest still had the lowest overall waveform errors: **0.0007950** Training CV and **0.0006771** Validation. Dilated TCN had approximately **5.6% higher Validation waveform error**, so the experiment does **not** demonstrate general superiority of deep learning. Validation had only four participants and was also used for diagnostics in earlier stages; it cannot serve as fully blind evidence for programme-wide decisions.

**Biomechanical trade-off.** Dilated TCN improved peak-magnitude MAE compared with Random Forest for both e1 (**0.0005855** versus **0.0006625**) and e3 (**0.0014333** versus **0.0016423**). However, Random Forest better located the original source-grid peak: respective e1/e3 peak-index errors were **1.417/1.005** for Random Forest and **1.615/1.495** for Dilated TCN. Lower peak-value error therefore does not imply better peak location. The peak-aware loss is a defined training objective, not a complete physics-informed biomechanical model.

**Recommended Stage 07 — Robustness, Peak-Error Trade-offs and Research Decision Evidence.** Quantify variation across new, predeclared Training-only seeds, compare Random Forest and temporal models by participant and activity, and assess whether peak-value improvements persist across folds without overfitting the repeatedly inspected Validation set. Notebook 08 uses the existing primary and secondary model recipes for the final Test assessment.
